# ViT NES experiments

This notebook tests the project's multi-output Flax Vision Transformer (`model="vit"`). It preserves the usual workflow: train → save JSON → plot later.

The local ViT is a multi-head adapter of the architecture shown in NetKet's ViT wave-function tutorial. It currently supports **2D** TFIM and toric-code inputs.

In [1]:
import sys
from pathlib import Path

# Works when Jupyter is launched from the project root or notebooks/.
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import jax
import jax.numpy as jnp
from nes_lattice.models import ModelSpec, init_model, apply_model
from nes_lattice.train import TrainConfig, train, save_history
from nes_lattice.plots import plot_history, plot_diagnostics, print_final

print("JAX devices:", jax.devices())
print("Project root:", PROJECT_ROOT)

JAX devices: [CudaDevice(id=0)]
Project root: /home/a/Anas.Roumeih/Desktop/Master Thesis/nes_lattice_project


E0701 20:56:49.155519     313 numa_hwloc.cc:121] Call to hwloc_set_cpubind() failed: Invalid argument [22]


## 1. ViT interface smoke test

The ViT receives flat spin configurations, reshapes them into 2D patches internally, and returns one real amplitude for each NES state.

In [2]:
spec = ModelSpec(
    model="vit",
    shape=(4, 4),
    k=2,
    n_sites=16,
    input_channels=1,
    vit_patch_size=2,
    vit_d_model=32,
    vit_num_layers=2,
    vit_num_heads=4,
    vit_mlp_ratio=2,
    dtype="float32",
)

params0 = init_model(jax.random.PRNGKey(0), spec)
test_spins = 2 * jax.random.bernoulli(jax.random.PRNGKey(1), 0.5, (5, 16)).astype(jnp.int8) - 1
psi = apply_model(params0, test_spins, spec)

print("input shape:", test_spins.shape)
print("output shape:", psi.shape)
print("all finite:", bool(jnp.all(jnp.isfinite(psi))))
print(psi)

input shape: (5, 16)
output shape: (5, 2)
all finite: True
[[1.7791525  5.20944   ]
 [1.4326271  0.66110504]
 [5.892685   1.4197489 ]
 [1.0418124  1.3479122 ]
 [0.2284495  0.30575812]]


## 2. Small correctness run: 2x2 TFIM

This is cheap enough for exact span evaluation. It is a smoke test for the sampler, ViT wrapper, JSON output, and plotting pipeline.

In [3]:
cfg = TrainConfig(
    shape=(2, 2),
    hamiltonian="tfim",
    k=2,
    J=1.0,
    g=1.0,
    model="vit",
    vit_patch_size=1,
    vit_d_model=32,
    vit_num_layers=2,
    vit_num_heads=4,
    vit_mlp_ratio=2,
    steps=2000,
    lr=5e-4,
    n_chains=64,
    n_samples=8,
    print_every=100,
    eval_exact_if_sites_leq=12,
    reference="auto",
    seed=0,
)

params, history = train(cfg)

save_path = PROJECT_ROOT / "results" / "sampled_nes_tfim_2x2_k2_vit.json"
save_history(history, save_path, cfg)
print("saved to:", save_path)

/home/a/Anas.Roumeih/venv/nqs311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: nk.optimizer.solver.nan_fallback(cheap, robust) retries a solve with the robust solver whenever NaN

appears.

/home/a/Anas.Roumeih/venv/nqs311/lib/python3.11/site-packages/netket/graph/common_lattices.py:126: InitializePeriodicLatticeOnSmallLatticeWarning: 
You are attempting to define a lattice with length 2 in dimension 0 using periodic boundary condition.

Lattice with less than two sites in one direction does not support periodic boundary condition.
The behavior of the lattice is equivalent to an open boundary condition in this direction.

To avoid this warning, consider either using a lattice with more than two sites in the direction you want to be periodic,
or define the graph using :class:`~netket.graph.Graph` by adding the edges manually.


-------------------------------------------------------
For more detailed informations, visit the following link:
	 https://netket.readthedocs.io/en/latest/api/_generated/errors/netket.errors.InitializePeriodicLatticeOnSmallLatticeWarning.html
or the list of all common errors and warnings at
	 https://netket.readthedocs.io/en/latest/api/errors.html


{'step': 0, 'loss_sum': -4.570687276126653, 'train_energy_estimator': nan, 'energies': [-4.818563460704514, 0.2478761845778613], 'reference': [-5.226251859505506, -4.82842712474619], 'reference_source': 'netket_lanczos_ed', 'abs_errors': [0.4076883988009925, 5.076303309324051], 'trace_error': 5.4839917081250436, 'condition_number_S': 16.850371493599017, 'sampler_accept_rate': nan, 'invalid_bundle_fraction': nan, 'grad_norm': nan, 'eval': {'S_min_eig': 21.423528051214863, 'S_max_eig': 360.9944063665098, 'S_rank': 2, 'S_floor': 1e-06, 'S_num_clipped': 0, 'S_eigenvalues': [21.423528051214863, 360.9944063665098], 'method': 'exact_span', 'accept_rate': None}}


TypeError: cond branches must have equal output types but they differ.

true_fun is <lambda> at /home/a/Anas.Roumeih/Desktop/Master Thesis/nes_lattice_project/src/nes_lattice/nes.py:80
false_fun is <lambda> at /home/a/Anas.Roumeih/Desktop/Master Thesis/nes_lattice_project/src/nes_lattice/nes.py:81

The output of true_fun has type float64[] but the corresponding output of false_fun has type float32[], so the dtypes do not match.

Revise true_fun and/or false_fun so that all output types match.

In [ ]:
print_final(save_path)
fig, ax = plot_history(save_path)
fig

In [ ]:
(fig_cond, ax_cond), (fig_diag, ax_diag) = plot_diagnostics(save_path)
fig_cond

In [ ]:
fig_diag

## 3. Main 2D TFIM run: 4x4

`4x4` has 16 spins, so the notebook uses sampled span evaluation by setting `eval_exact_if_sites_leq=12`. Start here after the 2x2 smoke test works.

In [ ]:
cfg = TrainConfig(
    shape=(4, 4),
    hamiltonian="tfim",
    k=2,
    J=1.0,
    g=1.0,
    model="vit",
    vit_patch_size=2,
    vit_d_model=64,
    vit_num_layers=2,
    vit_num_heads=4,
    vit_mlp_ratio=2,
    vit_use_positional_embeddings=True,
    steps=5000,
    lr=5e-4,
    n_chains=128,
    n_samples=8,
    print_every=100,
    eval_exact_if_sites_leq=12,
    eval_chains=128,
    eval_samples=32,
    reference="auto",
    seed=1,
)

params, history = train(cfg)

save_path = PROJECT_ROOT / "results" / "sampled_nes_tfim_4x4_k2_vit.json"
save_history(history, save_path, cfg)
print("saved to:", save_path)

In [ ]:
print_final(save_path)
fig, ax = plot_history(save_path)
fig

In [ ]:
(fig_cond, ax_cond), (fig_diag, ax_diag) = plot_diagnostics(save_path)
fig_cond

In [ ]:
fig_diag

## Notes

- `vit_patch_size` must divide both dimensions. For a `3x3` lattice use `vit_patch_size=1`.
- Start with `vit_d_model=64`, `vit_num_layers=2`, `vit_num_heads=4`.
- Use a lower learning rate than the CNN baseline: begin with `5e-4`; try `2e-4` if the determinant sampler becomes unstable.
- The ViT returns positive real amplitudes, which is appropriate for the stoquastic TFIM and toric-code tests in this project.